In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/pr_data.csv')

print(f" Loaded {len(df)} PRs")
print(f"Columns: {list(df.columns)}")
print(f"\nBug fix rate: {df['is_bug_fix'].mean():.1%}")

 Loaded 900 PRs
Columns: ['pr_number', 'title', 'body', 'additions', 'deletions', 'changed_files', 'commits', 'comments', 'review_comments', 'num_reviewers', 'author', 'reviewers', 'files_changed', 'diff_patch', 'is_bug_fix', 'created_at', 'merged_at', 'repo']

Bug fix rate: 74.7%


In [2]:
print(" CLEANING ")
before = len(df)

# Drop PRs with no diff patch
df = df[df['diff_patch'].notna()]
df = df[df['diff_patch'].str.len() > 20]

# Drop PRs with unknown author
df = df[df['author'] != 'unknown']

# Cap extreme outliers at 99th percentile
for col in ['additions', 'deletions', 'changed_files', 'commits']:
    cap = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=cap)

df = df.reset_index(drop=True)

print(f"Before cleaning : {before} PRs")
print(f"After cleaning  : {len(df)} PRs")
print(f"Dropped         : {before - len(df)} PRs")
print(f"Bug fix rate    : {df['is_bug_fix'].mean():.1%}")

 CLEANING 
Before cleaning : 900 PRs
After cleaning  : 900 PRs
Dropped         : 0 PRs
Bug fix rate    : 74.7%


In [3]:
print("STRUCTURAL FEATURES")

# Size features
df['total_changes']       = df['additions'] + df['deletions']
df['churn_ratio']         = df['deletions'] / (df['additions'] + 1)
df['avg_changes_per_file']= df['total_changes'] / (df['changed_files'] + 1)

# Complexity proxies
df['commits_per_file']    = df['commits'] / (df['changed_files'] + 1)
df['review_intensity']    = df['review_comments'] / (df['commits'] + 1)

# Reviewer features
df['has_reviewer']        = (df['num_reviewers'] > 0).astype(int)
df['solo_pr']             = (df['num_reviewers'] == 0).astype(int)
df['many_reviewers']      = (df['num_reviewers'] >= 3).astype(int)

# Time features
df['created_at'] = pd.to_datetime(df['created_at'])
df['merged_at']  = pd.to_datetime(df['merged_at'])
df['review_hours'] = (
    df['merged_at'] - df['created_at']
).dt.total_seconds() / 3600
df['fast_merge']  = (df['review_hours'] < 2).astype(int)
df['day_of_week'] = df['created_at'].dt.dayofweek
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

print(f" Added 12 structural features")
print(f"New columns: {['total_changes','churn_ratio','avg_changes_per_file','commits_per_file','review_intensity','has_reviewer','solo_pr','many_reviewers','review_hours','fast_merge','day_of_week','is_weekend']}")

STRUCTURAL FEATURES
 Added 12 structural features
New columns: ['total_changes', 'churn_ratio', 'avg_changes_per_file', 'commits_per_file', 'review_intensity', 'has_reviewer', 'solo_pr', 'many_reviewers', 'review_hours', 'fast_merge', 'day_of_week', 'is_weekend']


In [4]:
print("TEXT FEATURES")

# Combine title and body
df['text'] = df['title'].fillna('') + ' ' + df['body'].fillna('')

# Bug keyword count
bug_keywords = [
    'fix','bug','crash','error','null','exception',
    'fail','broken','issue','defect','regression',
    'patch','hotfix','revert','problem'
]
df['keyword_count'] = df['text'].apply(
    lambda x: sum(1 for k in bug_keywords if k in x.lower())
)

# Code diff features
df['diff_lines']      = df['diff_patch'].apply(
    lambda x: len(str(x).split('\n'))
)
df['has_test_file']   = df['files_changed'].apply(
    lambda x: int('test' in str(x).lower() or 'spec' in str(x).lower())
)
df['has_config_file'] = df['files_changed'].apply(
    lambda x: int(any(ext in str(x).lower() 
               for ext in ['.json','.yaml','.yml','.toml','.cfg']))
)
df['num_python_files']= df['files_changed'].apply(
    lambda x: str(x).lower().count('.py')
)
df['num_js_files']    = df['files_changed'].apply(
    lambda x: str(x).lower().count('.js') + str(x).lower().count('.ts')
)

# Title length
df['title_length'] = df['title'].fillna('').apply(len)

print(f"Added 8 text features")
print(f"\nKeyword count distribution:")
print(df['keyword_count'].value_counts().head(6))
print(f"\nPRs with test files: {df['has_test_file'].sum()} ({df['has_test_file'].mean():.1%})")

TEXT FEATURES
Added 8 text features

Keyword count distribution:
keyword_count
1    318
0    286
2    187
3     84
4     22
5      3
Name: count, dtype: int64

PRs with test files: 560 (62.2%)


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import scipy.sparse as sp

print("TF-IDF FEATURES")

tfidf = TfidfVectorizer(
    max_features  = 300,
    ngram_range   = (1, 2),
    token_pattern = r'\b[a-zA-Z_][a-zA-Z0-9_]{2,}\b',
    min_df        = 3
)

tfidf_matrix = tfidf.fit_transform(df['diff_patch'].fillna(''))

print(f" TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Top 20 tokens: {tfidf.get_feature_names_out()[:20]}")

# Save vectorizer
joblib.dump(tfidf, '../data/processed/tfidf_vectorizer.pkl')
print(" TF-IDF vectorizer saved")

TF-IDF FEATURES
 TF-IDF matrix shape: (900, 300)
Top 20 tokens: ['abort' 'act' 'action' 'actions' 'add' 'after' 'agent' 'all' 'already'
 'and' 'any' 'api' 'apimachinery' 'apimachinery pkg' 'apiserver' 'app'
 'append' 'are' 'arm64' 'array']
 TF-IDF vectorizer saved


In [6]:
print("=== ASSEMBLING FINAL FEATURES ===")

feature_cols = [
    # Size
    'total_changes', 'churn_ratio', 'avg_changes_per_file',
    'additions', 'deletions', 'changed_files',
    # Complexity
    'commits_per_file', 'review_intensity', 'commits',
    # Reviewer
    'has_reviewer', 'solo_pr', 'many_reviewers', 'num_reviewers',
    # Time
    'review_hours', 'fast_merge', 'day_of_week', 'is_weekend',
    # Text
    'keyword_count', 'diff_lines', 'has_test_file',
    'has_config_file', 'num_python_files', 'num_js_files',
    'title_length',
    # Engagement
    'comments', 'review_comments'
]

X = df[feature_cols].fillna(0)
y = df['is_bug_fix']

print(f"Feature matrix shape : {X.shape}")
print(f" Labels shape         : {y.shape}")
print(f" Features used        : {len(feature_cols)}")
print(f" Bug fix rate         : {y.mean():.1%}")
print(f"\nFeature list:\n{feature_cols}")

=== ASSEMBLING FINAL FEATURES ===
Feature matrix shape : (900, 26)
 Labels shape         : (900,)
 Features used        : 26
 Bug fix rate         : 74.7%

Feature list:
['total_changes', 'churn_ratio', 'avg_changes_per_file', 'additions', 'deletions', 'changed_files', 'commits_per_file', 'review_intensity', 'commits', 'has_reviewer', 'solo_pr', 'many_reviewers', 'num_reviewers', 'review_hours', 'fast_merge', 'day_of_week', 'is_weekend', 'keyword_count', 'diff_lines', 'has_test_file', 'has_config_file', 'num_python_files', 'num_js_files', 'title_length', 'comments', 'review_comments']


In [7]:
from sklearn.preprocessing import StandardScaler

print("TRAIN / TEST SPLIT")

# Sort by date — never use random split for time series data
df_sorted = df.sort_values('created_at').reset_index(drop=True)
X_sorted  = X.loc[df_sorted.index]
y_sorted  = y.loc[df_sorted.index]

split_idx = int(len(df_sorted) * 0.8)

X_train = X_sorted.iloc[:split_idx]
X_test  = X_sorted.iloc[split_idx:]
y_train = y_sorted.iloc[:split_idx]
y_test  = y_sorted.iloc[split_idx:]

print(f"Train size : {len(X_train)} PRs ({len(X_train)/len(df):.0%})")
print(f"Test size  : {len(X_test)} PRs ({len(X_test)/len(df):.0%})")
print(f"Train bug rate : {y_train.mean():.1%}")
print(f"Test bug rate  : {y_test.mean():.1%}")

# Scale features
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"\nFeatures scaled with StandardScaler")

TRAIN / TEST SPLIT
Train size : 720 PRs (80%)
Test size  : 180 PRs (20%)
Train bug rate : 70.4%
Test bug rate  : 91.7%

Features scaled with StandardScaler


In [8]:
print(" SAVING PROCESSED DATA ")

# Save feature matrix and labels
X.to_csv('../data/processed/X_features.csv', index=False)
y.to_csv('../data/processed/y_labels.csv', index=False)

# Save train/test splits
pd.DataFrame(X_train_s, columns=feature_cols).to_csv(
    '../data/processed/X_train.csv', index=False)
pd.DataFrame(X_test_s, columns=feature_cols).to_csv(
    '../data/processed/X_test.csv', index=False)
y_train.reset_index(drop=True).to_csv(
    '../data/processed/y_train.csv', index=False)
y_test.reset_index(drop=True).to_csv(
    '../data/processed/y_test.csv', index=False)

# Save scaler
joblib.dump(scaler, '../data/processed/scaler.pkl')

# Save cleaned full dataframe
df.to_csv('../data/processed/pr_cleaned.csv', index=False)

print(" Saved files:")
print("   data/processed/X_features.csv")
print("   data/processed/y_labels.csv")
print("   data/processed/X_train.csv")
print("   data/processed/X_test.csv")
print("   data/processed/y_train.csv")
print("   data/processed/y_test.csv")
print("   data/processed/scaler.pkl")
print("   data/processed/pr_cleaned.csv")
print("   data/processed/tfidf_vectorizer.pkl")

 SAVING PROCESSED DATA 
 Saved files:
   data/processed/X_features.csv
   data/processed/y_labels.csv
   data/processed/X_train.csv
   data/processed/X_test.csv
   data/processed/y_train.csv
   data/processed/y_test.csv
   data/processed/scaler.pkl
   data/processed/pr_cleaned.csv
   data/processed/tfidf_vectorizer.pkl


In [9]:
print("=== TASK 3 COMPLETE ===")
print(f" PRs after cleaning          : {len(df)}")
print(f" Total features engineered   : {len(feature_cols)}")
print(f" Train samples               : {len(X_train)}")
print(f" Test samples                : {len(X_test)}")
print(f" Files saved in processed/   : 9")


=== TASK 3 COMPLETE ===
 PRs after cleaning          : 900
 Total features engineered   : 26
 Train samples               : 720
 Test samples                : 180
 Files saved in processed/   : 9
